# 0 Libraries inladen

In [151]:
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import random

env = gym.make("Blackjack-v1")

sample_size = 1000 # voor testen op 1000, voor werkelijk berekenen op 1000000
tokens = 1 # het aantal tokens om in te zetten tijdens het spelen van blackjack

# 1 Opzetten spel

In [152]:
def blackjack(keuze_strategie, inzet): 
    obs, info = env.reset() # obs bevat de speler's som, de dealer's zichtbare kaart, en of de speler een usable ace heeft
    terminated = False # of de ronde is afgelopen
    truncated = False # of de ronde is afgebroken (bijvoorbeeld door een time limit)
    
    while not (terminated or truncated): # zolang de ronde niet is afgelopen of afgebroken
        speler_som = obs[0] # de som van de speler's kaarten
        dealer_kaarten_zichtbaar = [env.unwrapped.dealer[0]] # de zichtbare kaart van de dealer
        speler_kaarten = env.unwrapped.player # de kaarten van de speler
        
        # print(f"Speler's som: {speler_som}, Dealer's zichtbare kaart: {dealer_kaarten_zichtbaar[0]}, Speler's kaarten: {speler_kaarten}") # print de huidige situatie van de speler en dealer
        
        keuze = keuze_strategie(speler_som, dealer_kaarten_zichtbaar, speler_kaarten) # de strategie bepaalt of de speler 'hit' of 'stick' kiest op basis van de huidige situatie
        
        if keuze == 'hit': 
            action = 1 # actie 1 betekent 'hit' in de Blackjack omgeving
        elif keuze == 'stick':
            action = 0 # actie 0 betekent 'stick' in de Blackjack omgeving
            
        obs, reward, terminated, truncated, info = env.step(action) # voer de gekozen actie uit in de omgeving en ontvang de nieuwe observatie, beloning, en of de ronde is afgelopen of afgebroken
        
    if reward > 0:
        if speler_kaarten == [1, 10]:  # Blackjack
            winst = inzet * 2.5 # bij een Blackjack wint de speler 1.5 keer zijn inzet als winst (plus de inzet terug), wat neerkomt op een totale uitbetaling van 2.5 keer de inzet
            resultaat = 'Blackjack'
        else:
            winst = inzet * 2 # bij een normale overwinning wint de speler zijn inzet terug plus een gelijk bedrag als winst
            resultaat = 'Gewonnen'
    elif reward < 0:
        winst = 0 # bij verlies verliest de speler zijn inzet
        resultaat = 'Verloren'
    else:
        winst = inzet # bij gelijkspel is de winst nul
        resultaat = 'Gelijk'
        
    return winst, resultaat

# 2 Regels van Blackjack

Blackjack is een kaartspel wat veel gespeeld wordt in casino's. De speler speelt niet tegen andere spelers, maar tegen de bank. Het doel van Blackjack is om zo dicht bij de 21 punten te halen zonder er overheen te gaan. Als de speler dichterbij de 21 komt dan de bank, dan wint de speler. Als de bank dichterbij komt wint de bank (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014).

Voor dit portfolio is de volledige casino werking van dit spel niet relevant, hierbij gaat het dan over het plaatsen van een inzet en deze verdubbelen bij een goede hand.

## 2.1 De kaarten delen
Zodra de speler de inzet geplaatst heeft (of in het geval van dit porfolio als het spel geïnitialiseerd wordt), worden de kaarten gedeeld. De speler krijgt twee zichtbare kaarten. De bank krijgt eveneens twee kaarten, waarvan er één zichtbaar is voor de speler (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014).

## 2.2 Het spel
Zodra de kaarten gedeeld heeft de speler twee keuzes: 'hit' en 'stick'. Als de speler kiest voor 'hit' dan krijgt de speler een nieuwe kaart, als de speler kiest voor 'stick' dan is het spel voor de speler afgelopen. In het casino zijn er ook de opties 'split', 'double down', 'surrender', echter zijn deze niet in de gymnasium library versie van Blackjack (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014).

## 2.3 De dealer
Als de speler klaar is met zijn spel, gaat de dealer zijn spel spelen. Hiervoor zijn twee vaste regels:

1. Als de bank minder dan 17 punten heeft moet er 'hit' gespeeld worden.
2. Als de bank 17 punten of meer heeft moet er 'stand' gespeeld worden.

## 2.4 Uitbetaling
Als het spel is afgelopen volgt de volgende uitbetaling (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014):

- Winst speler zonder Blackjack (exact 21 punten): 1x de inzet.
- Gelijkspel: inzet blijft staan voor de volgende ronde.
- Verlies: verlies inzet.
- Winst met Blackjack: 1.5x de inzet.

## 2.5 Waardes van de kaarten
In het Blackjack worden de volgende kaartwaardes gebruikt (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014):

- 2 t/m 10: eigen waarde.
- Aas: 1 of 11 punten (welke de som dichterbij de 21 krijgt).
- Boer, Vrouw en Heer: 10 punten.

# 3 Uitleg gekozen strategieën

# 4 Strategieën uitwerken en uitvoeren

In [153]:
def baseline_strat(speler_som, dealer_kaarten_zichtbaar, speler_kaarten):
    return random.choice(['hit', 'stick']) # deze strategie kiest willekeurig tussen 'hit' en 'stick', ongeacht de situatie

baseline_results = [] 
baseline_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = baseline_tokens * 0.5 # de speler zet 50% van zijn tokens in voor elke ronde
    baseline_tokens -= inzet # trek de inzet af van de totale tokens voordat de ronde begint
    resultaat = blackjack(baseline_strat, inzet)
    baseline_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    baseline_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de baseline_results lijst
    
print(f"Na {sample_size} rondes met de baseline strategie heeft de speler {baseline_tokens} tokens over.")

Na 1000 rondes met de baseline strategie heeft de speler 1.7927334462766999e-150 tokens over.


In [154]:
def strat1_boekje(speler_som, dealer_kaarten_zichtbaar, speler_kaarten):
    if speler_som == 21:  # Blackjack
        return 'stick' 
    elif 1 not in speler_kaarten: # als de speler geen Ace heeft, dan is de strategie gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer
        if dealer_kaarten_zichtbaar[0] in [2, 3]: # als de dealer een 2 of 3 heeft, is het beter om te 'hit' als de speler's som 12 of minder is, omdat de kans groot is dat de dealer zal busten (boven 21 gaat)
            if speler_som <= 12:
                return 'hit'
            else:
                return 'stick'
        elif dealer_kaarten_zichtbaar[0] in [4, 5, 6]: # als de dealer een 4, 5, of 6 heeft, is het beter om te 'hit' als de speler's som 11 of minder is, omdat de kans groot is dat de dealer zal busten
            if speler_som <= 11:
                return 'hit'
            else:
                return 'stick'
        elif dealer_kaarten_zichtbaar[0] in [7, 8, 9, 10, 1]: # als de dealer een 7, 8, 9, 10, of Ace heeft, is het beter om te 'hit' als de speler's som 16 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if speler_som <= 16:
                return 'hit'
            else:
                return 'stick'
    else: # als de speler een Ace heeft, dan kan de Ace als 1 of 11 worden geteld, dus de strategie is gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer, maar met een hogere drempel voor 'hit' omdat de speler meer flexibiliteit heeft met de Ace
        if dealer_kaarten_zichtbaar[0] in [2, 3, 4, 5, 6, 7, 8]: # als de dealer een 2, 3, 4, 5, 6, 7, of 8 heeft, is het beter om te 'hit' als de speler's som (met de Ace geteld als 11) 17 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6]):
                return 'hit'
            else:
                return 'stick'
        elif dealer_kaarten_zichtbaar[0] in [9, 10, 1]: # als de dealer een 9, 10, of Ace heeft, is het beter om te 'hit' als de speler's som (met de Ace geteld als 11) 18 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6, 7]):
                return 'hit'
            else:
                return 'stick'
        
    
strat1_results = []
strat1_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat1_tokens * 0.5 # de speler zet 50% van zijn tokens in voor elke ronde
    strat1_tokens -= inzet # trek de inzet af van de totale tokens voordat de ronde begint
    resultaat = blackjack(strat1_boekje, inzet)
    strat1_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat1_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat1_results lijst
    
print(f"Na {sample_size} rondes met de boekje strategie heeft de speler {strat1_tokens} tokens over.")

Na 1000 rondes met de boekje strategie heeft de speler 2.8644948944039864e-80 tokens over.


In [155]:
def strat2_boekje_invers(speler_som, dealer_kaarten_zichtbaar, speler_kaarten):
    if speler_som == 21:  # Blackjack
        return 'stick'
    elif 1 not in speler_kaarten: # als de speler geen Ace heeft, dan is de strategie gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer, maar met omgekeerde beslissingen in vergelijking met de boekje strategie
        if dealer_kaarten_zichtbaar[0] in [2, 3]: # als de dealer een 2 of 3 heeft, is het beter om te 'stick' als de speler's som 12 of minder is, omdat de kans groot is dat de dealer zal busten (boven 21 gaat)
            if speler_som <= 12:
                return 'stick'
            else:
                return 'hit'
        elif dealer_kaarten_zichtbaar[0] in [4, 5, 6]: # als de dealer een 4, 5, of 6 heeft, is het beter om te 'stick' als de speler's som 11 of minder is, omdat de kans groot is dat de dealer zal busten
            if speler_som <= 11:
                return 'stick'
            else:
                return 'hit'
        elif dealer_kaarten_zichtbaar[0] in [7, 8, 9, 10, 1]: # als de dealer een 7, 8, 9, 10, of Ace heeft, is het beter om te 'stick' als de speler's som 16 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if speler_som <= 16:
                return 'stick'
            else:
                return 'hit'
    else: # als de speler een Ace heeft, dan kan de Ace als 1 of 11 worden geteld, dus de strategie is gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer, maar met een hogere drempel voor 'stick' omdat de speler meer flexibiliteit heeft met de Ace, en met omgekeerde beslissingen in vergelijking met de boekje strategie
        if dealer_kaarten_zichtbaar[0] in [2, 3, 4, 5, 6, 7, 8]: # als de dealer een 2, 3, 4, 5, 6, 7, of 8 heeft, is het beter om te 'stick' als de speler's som (met de Ace geteld als 11) 17 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6]):
                return 'stick'
            else:
                return 'hit'
        elif dealer_kaarten_zichtbaar[0] in [9, 10, 1]: # als de dealer een 9, 10, of Ace heeft, is het beter om te 'stick' als de speler's som (met de Ace geteld als 11) 18 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6, 7]):
                return 'stick'
            else:
                return 'hit'
        
    
strat2_results = []
strat2_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat2_tokens * 0.5 # de speler zet 50% van zijn tokens in voor elke ronde
    strat2_tokens -= inzet # trek de inzet af van de totale tokens voordat de ronde begint
    resultaat = blackjack(strat2_boekje_invers, inzet)
    strat2_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat2_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat2_results lijst

print(f"Na {sample_size} rondes met de inverse strategie van het boekje heeft de speler {strat2_tokens} tokens over.")

Na 1000 rondes met de inverse strategie van het boekje heeft de speler 3.532867356317606e-195 tokens over.


In [156]:
def strat3_hit_17(speler_som, dealer_kaarten_zichtbaar, speler_kaarten):
    if speler_som == 21:  # Blackjack
        return 'stick'
    elif speler_som <= 17: # deze strategie kiest altijd 'hit' als de som van de speler's kaarten 17 of minder is, ongeacht de zichtbare kaart van de dealer of de specifieke kaarten van de speler, en kiest 'stick' als de som 18 of meer is
        return 'hit'
    else:
        return 'stick'
        
    
strat3_results = []
strat3_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat3_tokens * 0.5 # de speler zet 50% van zijn tokens in voor elke ronde
    strat3_tokens -= inzet # trek de inzet af van de totale tokens voordat de ronde begint
    resultaat = blackjack(strat3_hit_17, inzet)
    strat3_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat3_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat3_results lijst

print(f"Na {sample_size} rondes met de hit tot 17 strategie heeft de speler {strat3_tokens} tokens over.")

Na 1000 rondes met de hit tot 17 strategie heeft de speler 2.9778802998473535e-75 tokens over.


In [157]:
def strat4_stand_12(speler_som, dealer_kaarten_zichtbaar, speler_kaarten):
    if speler_som == 21:  # Blackjack
        return 'stick'
    elif speler_som >= 12: # deze strategie kiest altijd 'stick' als de som van de speler's kaarten 12 of meer is, ongeacht de zichtbare kaart van de dealer of de specifieke kaarten van de speler, en kiest 'hit' als de som 11 of minder is
        return 'stick'
    else:
        return 'hit'
        
    
strat4_results = []
strat4_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat4_tokens * 0.5 # de speler zet 50% van zijn tokens in voor elke ronde
    strat4_tokens -= inzet # trek de inzet af van de totale tokens voordat de ronde begint
    resultaat = blackjack(strat4_stand_12, inzet)
    strat4_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat4_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat4_results lijst

print(f"Na {sample_size} rondes met de stand op 12 strategie heeft de speler {strat4_tokens} tokens over.")

Na 1000 rondes met de stand op 12 strategie heeft de speler 1.1883142436827011e-80 tokens over.


In [158]:
def strat5_dealer_weakness(speler_som, dealer_kaarten_zichtbaar, speler_kaarten):
    if speler_som == 21:  # Blackjack
        return 'stick'
    elif dealer_kaarten_zichtbaar[0] in [2, 3, 4, 5, 6]: # deze strategie kiest 'hit' als de dealer een zwakke kaart heeft (2, 3, 4, 5, of 6)
        return 'stick'
    elif dealer_kaarten_zichtbaar[0] in [7, 8, 9, 10, 1]: # deze strategie kiest 'hit' als de dealer een sterke kaart heeft (7, 8, 9, 10, of Ace)
        if speler_som <= 21:
            return 'hit'
        else:
            return 'stick'
        
    
strat5_results = []
strat5_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat5_tokens * 0.5 # de speler zet 50% van zijn tokens in voor elke ronde
    strat5_tokens -= inzet # trek de inzet af van de totale tokens voordat de ronde begint
    resultaat = blackjack(strat5_dealer_weakness, inzet)
    strat5_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat5_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat5_results lijst

print(f"Na {sample_size} rondes met de dealer weakness strategie heeft de speler {strat5_tokens} tokens over.")

Na 1000 rondes met de dealer weakness strategie heeft de speler 1.1172196103947552e-167 tokens over.


In [159]:
def strat6_always_hit(speler_som, dealer_kaarten_zichtbaar, speler_kaarten):
    if speler_som != 21:  # deze strategie kiest altijd 'hit' als de som van de speler's kaarten niet 21 is, ongeacht de zichtbare kaart van de dealer of de specifieke kaarten van de speler, en kiest 'stick' alleen als de speler al een Blackjack heeft
        return 'hit'
    else:
        return 'stick'
        
    
strat6_results = []
strat6_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat6_tokens * 0.5 # de speler zet 50% van zijn tokens in voor elke ronde
    strat6_tokens -= inzet # trek de inzet af van de totale tokens voordat de ronde begint
    resultaat = blackjack(strat6_always_hit, inzet)
    strat6_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat6_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat6_results lijst

print(f"Na {sample_size} rondes met de always hit strategie heeft de speler {strat6_tokens} tokens over.")

Na 1000 rondes met de always hit strategie heeft de speler 2.3344845196309282e-220 tokens over.


# 5 Strategieën vergelijken

In [164]:
df = pd.DataFrame({
    'Strategie': ['Random (Baseline)', 'Boekje', 'Boekje Invers', 'Hit tot 17', 'Stand vanaf 12', 'Dealer Weakness', 'Always Hit'],
    'Blackjack (%)': [round(baseline_results.count('Blackjack') / sample_size * 100, 1), round(strat1_results.count('Blackjack') / sample_size * 100, 1), round(strat2_results.count('Blackjack') / sample_size * 100, 1), round(strat3_results.count('Blackjack') / sample_size * 100, 1), round(strat4_results.count('Blackjack') / sample_size * 100, 1), round(strat5_results.count('Blackjack') / sample_size * 100, 1), round(strat6_results.count('Blackjack') / sample_size * 100, 1)],
    'Gewonnen (%)': [round((baseline_results.count('Gewonnen') + baseline_results.count('Blackjack')) / sample_size * 100, 1), round((strat1_results.count('Gewonnen') + strat1_results.count('Blackjack')) / sample_size * 100, 1), round((strat2_results.count('Gewonnen') + strat2_results.count('Blackjack')) / sample_size * 100, 1), round((strat3_results.count('Gewonnen') + strat3_results.count('Blackjack')) / sample_size * 100, 1), round((strat4_results.count('Gewonnen') + strat4_results.count('Blackjack')) / sample_size * 100, 1), round((strat5_results.count('Gewonnen') + strat5_results.count('Blackjack')) / sample_size * 100, 1), round((strat6_results.count('Gewonnen') + strat6_results.count('Blackjack')) / sample_size * 100, 1)],
    'Verloren (%)': [round(baseline_results.count('Verloren') / sample_size * 100, 1), round(strat1_results.count('Verloren') / sample_size * 100, 1), round(strat2_results.count('Verloren') / sample_size * 100, 1), round(strat3_results.count('Verloren') / sample_size * 100, 1), round(strat4_results.count('Verloren') / sample_size * 100, 1), round(strat5_results.count('Verloren') / sample_size * 100, 1), round(strat6_results.count('Verloren') / sample_size * 100, 1)],
    'Gelijk (%)': [round(baseline_results.count('Gelijk') / sample_size * 100, 1), round(strat1_results.count('Gelijk') / sample_size * 100, 1), round(strat2_results.count('Gelijk') / sample_size * 100, 1), round(strat3_results.count('Gelijk') / sample_size * 100, 1), round(strat4_results.count('Gelijk') / sample_size * 100, 1), round(strat5_results.count('Gelijk') / sample_size * 100, 1), round(strat6_results.count('Gelijk') / sample_size * 100, 1)],
    'Overgebleven Tokens': [round(baseline_tokens, 0), round(strat1_tokens, 0), round(strat2_tokens, 0), round(strat3_tokens, 0), round(strat4_tokens, 0), round(strat5_tokens, 0), round(strat6_tokens, 0)],
    'Tokens verlies (%)': [round((tokens - baseline_tokens) / tokens * 100, 1), round((tokens - strat1_tokens) / tokens * 100, 1), round((tokens - strat2_tokens) / tokens * 100, 1), round((tokens - strat3_tokens) / tokens * 100, 1), round((tokens - strat4_tokens) / tokens * 100, 1), round((tokens - strat5_tokens) / tokens * 100, 1), round((tokens - strat6_tokens) / tokens * 100, 1)]
})

In [165]:
df

,Strategie,Blackjack (%),Gewonnen (%),Verloren (%),Gelijk (%),Overgebleven Tokens,Tokens verlies (%)
0,Random (Baseline),1.2,28.7,66.8,4.5,0.0,100.0
1,Boekje,2.7,41.5,51.3,7.2,0.0,100.0
2,Boekje Invers,2.0,21.3,77.5,1.2,0.0,100.0
3,Hit tot 17,2.1,41.5,49.5,9.0,0.0,100.0
4,Stand vanaf 12,1.9,42.1,51.6,6.3,0.0,100.0
5,Dealer Weakness,2.2,25.9,71.1,3.0,0.0,100.0
6,Always Hit,2.5,15.7,82.7,1.6,0.0,100.0


# Literatuurlijst

- *Blackjack spelregels - Regels voor Blackjack | TOTO.* (z.d.). Geraadpleegd op 12 maart 2026, van https://www.toto.nl/klantenservice/spelregels-voor-blackjack
- *Blackjack: uitleg & spelregels.* (2025, 4 november). Club BetCity. Geraadpleegd op 12 maart 2026, van https://club.betcity.nl/blackjack/regels
- Spelregels.Eu. (2020, 20 april). *Hoe speel je Blackjack.* spelregels.eu. Geraadpleegd op 12 maart 2026, van https://www.spelregels.eu/blackjack
- Van Geest, M. (2014). *Blackjack: regels en uitleg.* Meneer Casino. Geraadpleegd op 12 maart 2026, van https://meneercasino.com/online-casino-tips/blackjack-regels-en-uitleg